# 007: Ensemble Baseline for AI Text Detection

Combines **LR (001)**, **DeBERTa (003)**, and **Qwen LoRA (006)** into ensemble approaches.

Evaluated on PAN2025 validation + HC3 + RAID. Three ensemble strategies:
- **Mean**: Equal-weight probability averaging
- **Weighted**: Weights proportional to prior OOD ROC-AUC
- **Majority Vote**: Hard voting at 0.5 threshold

**Requirements**: GPU runtime (T4 or better).

In [12]:
# =====================================================
# CELL 0 — Environment setup
# =====================================================
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA L4


In [13]:
# =====================================================
# CELL 1 — Install dependencies
# =====================================================
!pip install -q peft gdown scikit-learn 'datasets<3.0'

In [14]:
# =====================================================
# CELL 2 — Download datasets
# =====================================================
import json, random, requests
import gdown
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)

def save_jsonl(records, path):
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'  Saved {len(records)} records to {path.name}')

# --- PAN2025 Train (needed to retrain LR) ---
print('Downloading PAN2025 train...')
train_id = '1k84YY0p8JTuTE40yNEkgzsdaSxt_1nl-'
gdown.download(
    f'https://drive.google.com/uc?id={train_id}',
    str(DATA_DIR / 'train.jsonl'), quiet=True
)
print('  Done.')

# --- PAN2025 Validation ---
print('Downloading PAN2025 val...')
val_id = '12szC1TcNPN9KULPNjTZsEyG8RafxfsJy'
gdown.download(
    f'https://drive.google.com/uc?id={val_id}',
    str(DATA_DIR / 'val.jsonl'), quiet=True
)
print('  Done.')

# --- HC3 Wiki ---
print('\nDownloading HC3 (wiki subset)...')
hc3_url = 'https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/wiki_csai.jsonl'
resp = requests.get(hc3_url)
resp.raise_for_status()
records = []
for line in resp.text.strip().split('\n'):
    row = json.loads(line)
    for ans in row.get('human_answers', []):
        if ans.strip():
            records.append({'text': ans.strip(), 'label': 0, 'source': 'hc3_human'})
    for ans in row.get('chatgpt_answers', []):
        if ans.strip():
            records.append({'text': ans.strip(), 'label': 1, 'source': 'hc3_chatgpt'})
save_jsonl(records, DATA_DIR / 'hc3_wiki.jsonl')

# --- RAID (streaming) ---
print('\nDownloading RAID (streaming)...')
raid = load_dataset('liamdugan/raid', 'raid', split='train', streaming=True)
human_recs, ai_recs = [], []
for row in raid:
    if row.get('language', 'en') != 'en':
        continue
    text = row.get('generation', '')
    if not text or len(text.strip()) < 50:
        continue
    label = 0 if row.get('model') == 'human' else 1
    rec = {'text': text.strip(), 'label': label, 'source': 'raid'}
    if label == 0:
        human_recs.append(rec)
    else:
        ai_recs.append(rec)
    if len(human_recs) >= 1500 and len(ai_recs) >= 1500:
        break
n = min(1000, len(human_recs), len(ai_recs))
records = random.sample(human_recs, n) + random.sample(ai_recs, n)
random.shuffle(records)
save_jsonl(records, DATA_DIR / 'raid.jsonl')

# --- Summary ---
print('\n=== Dataset Summary ===')
for f in sorted(DATA_DIR.glob('*.jsonl')):
    with open(f) as fh:
        lines = fh.readlines()
    labels = [json.loads(l)['label'] for l in lines]
    n0, n1 = labels.count(0), labels.count(1)
    print(f'  {f.name:30s}  {len(lines):>6d} samples  (human={n0}, ai={n1})')

  Done.
  Done.

  Saved 1684 records to hc3_wiki.jsonl

  Saved 2000 records to raid.jsonl

=== Dataset Summary ===
  hc3_wiki.jsonl                    1684 samples  (human=842, ai=842)
  raid.jsonl                        2000 samples  (human=1000, ai=1000)
  train.jsonl                      23707 samples  (human=9101, ai=14606)
  val.jsonl                         3589 samples  (human=1277, ai=2312)


In [15]:
# =====================================================
# CELL 3 — Utilities (inlined: chunking, metrics)
# =====================================================
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score, fbeta_score
from tqdm import tqdm

CHUNK_SIZE = 512
CHUNK_STRIDE = 256


def chunk_text_by_tokens(input_ids, attention_mask, chunk_size=512,
                         stride=256, pad_token_id=0):
    if len(input_ids) <= chunk_size:
        return [input_ids], [attention_mask]
    chunked_ids, chunked_masks = [], []
    start = 0
    while start < len(input_ids):
        end = min(start + chunk_size, len(input_ids))
        c_ids = input_ids[start:end]
        c_mask = attention_mask[start:end]
        if len(c_ids) < chunk_size:
            pad_len = chunk_size - len(c_ids)
            c_ids = c_ids + [pad_token_id] * pad_len
            c_mask = c_mask + [0] * pad_len
        chunked_ids.append(c_ids)
        chunked_masks.append(c_mask)
        start += stride
        if start + stride >= len(input_ids):
            break
    return chunked_ids, chunked_masks


def get_all_metrics(y_true, y_pred, y_prob):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    n = len(y_true)
    unanswered = (y_prob == 0.5)
    n_u = np.sum(unanswered)
    n_c = np.sum((y_pred == y_true) & (~unanswered))
    c_at_1 = (1/n) * (n_c + n_u * (n_c / n)) if n > 0 else 0
    mod_pred = y_pred.copy()
    mod_pred[unanswered] = 0
    return {
        'ROC-AUC': float(roc_auc_score(y_true, y_prob)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'F1': float(f1_score(y_true, y_pred)),
        'C@1': float(c_at_1),
        'F0.5u': float(fbeta_score(y_true, mod_pred, beta=0.5)),
    }


def predict_one_chunked(model, tokenizer, text, device):
    """Predict P(AI) for a single text with chunking."""
    enc = tokenizer(text, truncation=False, return_tensors='pt', add_special_tokens=True)
    ids = enc['input_ids'][0].tolist()
    mask = enc['attention_mask'][0].tolist()
    c_ids, c_masks = chunk_text_by_tokens(
        ids, mask, chunk_size=CHUNK_SIZE, stride=CHUNK_STRIDE,
        pad_token_id=tokenizer.pad_token_id or 0,
    )
    probs = []
    with torch.no_grad():
        for ci, cm in zip(c_ids, c_masks):
            inp = {
                'input_ids': torch.tensor([ci]).to(device),
                'attention_mask': torch.tensor([cm]).to(device),
            }
            out = model(**inp)
            p = torch.softmax(out.logits.float(), dim=1)
            probs.append(p[0, 1].item())
    return float(np.mean(probs))


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


print('Utilities loaded.')

Utilities loaded.


## Load Datasets

In [16]:
# =====================================================
# CELL 4 — Load all datasets into memory
# =====================================================
DATASETS = {
    'PAN2025': DATA_DIR / 'val.jsonl',
    'HC3': DATA_DIR / 'hc3_wiki.jsonl',
    'RAID': DATA_DIR / 'raid.jsonl',
}

dataset_cache = {}
for name, path in DATASETS.items():
    if not path.exists():
        print(f'WARNING: {name} not found at {path}')
        continue
    records = load_jsonl(str(path))
    texts = [r['text'] for r in records]
    labels = np.array([r['label'] for r in records])
    dataset_cache[name] = {'texts': texts, 'labels': labels}
    print(f'{name}: {len(texts)} samples, {labels.mean():.1%} AI')

PAN2025: 3589 samples, 64.4% AI
HC3: 1684 samples, 50.0% AI
RAID: 2000 samples, 50.0% AI


## Load Models

### Model 1: Logistic Regression (001)

Retrained inline from PAN2025 training data. Matches experiment 001 exactly:
TF-IDF (10k features, 1-2 ngrams) + LogisticRegression (C=1.0).

In [17]:
# =====================================================
# CELL 5 — Train LR model from PAN2025 training data
# =====================================================
# Retrain inline (~5 seconds) instead of loading from
# a private repo. Matches experiment 001 exactly.

import re, string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Training LR model from PAN2025 train.jsonl...')
train_records = load_jsonl(str(DATA_DIR / 'train.jsonl'))
train_texts = [clean_text(r['text']) for r in train_records]
train_labels = [r['label'] for r in train_records]

lr_vectorizer = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2), stop_words='english'
)
X_train = lr_vectorizer.fit_transform(train_texts)
print(f'  TF-IDF features: {X_train.shape}')

lr_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
lr_model.fit(X_train, train_labels)
print(f'  LR trained on {len(train_labels)} samples')

def lr_predict_proba(texts):
    cleaned = [clean_text(t) for t in texts]
    X = lr_vectorizer.transform(cleaned)
    return lr_model.predict_proba(X)[:, 1]

# Quick test
test_prob = lr_predict_proba(['This is a test.'])
print(f'LR ready. Test P(AI): {test_prob[0]:.4f}')

Training LR model from PAN2025 train.jsonl...
  TF-IDF features: (23707, 10000)
  LR trained on 23707 samples
LR ready. Test P(AI): 0.8380


### Model 2: DeBERTa (003)

In [19]:
# =====================================================
# CELL 6 — Load DeBERTa from HuggingFace
# =====================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEBERTA_HF = 'hersheys-baklava/deberta-pan2026'

deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_HF)
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_HF)
deberta_model.to(device)
deberta_model.eval()

def deberta_predict_proba(texts):
    """Get P(AI) from DeBERTa for a list of texts."""
    probs = []
    for text in tqdm(texts, desc='DeBERTa', leave=False):
        probs.append(predict_one_chunked(deberta_model, deberta_tokenizer, text, device))
    return np.array(probs)

# Quick test
test_prob = predict_one_chunked(deberta_model, deberta_tokenizer, 'This is a test.', device)
print(f'DeBERTa loaded from {DEBERTA_HF}. Test P(AI): {test_prob:.4f}')

config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

DeBERTa loaded from hersheys-baklava/deberta-pan2026. Test P(AI): 0.2706


### Model 3: Qwen LoRA (006)

In [23]:
# =====================================================
# CELL 7 — Load model + LoRA adapter
# =====================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from peft import PeftModel

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
ADAPTER_REPO = 'hersheys-baklava/qwen-lora-pan2026'

print(f'Loading base model: {MODEL_NAME}')
config = AutoConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = config.eos_token_id
config.num_labels = 2
config.problem_type = 'single_label_classification'

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config, torch_dtype=torch.bfloat16,
)

print(f'Loading LoRA adapter from: {ADAPTER_REPO}')
qwen_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
qwen_model = qwen_model.merge_and_unload()
qwen_model.to(device)
qwen_model.eval()

qwen_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
qwen_tokenizer.padding_side = 'left'

print(f'Model loaded on {device}. Params: {sum(p.numel() for p in qwen_model.parameters()):,}')

def qwen_predict_proba(texts):
    """Get P(AI) from Qwen LoRA for a list of texts."""
    probs = []
    for text in tqdm(texts, desc='Qwen LoRA', leave=False):
        probs.append(predict_one_chunked(qwen_model, qwen_tokenizer, text, device))
    return np.array(probs)

# Quick test
test_prob = predict_one_chunked(qwen_model, qwen_tokenizer, 'This is a test.', device)
print(f'Qwen LoRA loaded and merged. Test P(AI): {test_prob:.4f}')

Loading base model: Qwen/Qwen2.5-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading LoRA adapter from: hersheys-baklava/qwen-lora-pan2026
Model loaded on cuda. Params: 1,543,717,376
Qwen LoRA loaded and merged. Test P(AI): 0.0005


## Generate Per-Model Predictions

In [24]:
# =====================================================
# CELL 8 — Run all models on all datasets
# =====================================================
MODEL_PREDICTORS = {
    'LR': lr_predict_proba,
    'DeBERTa': deberta_predict_proba,
    'Qwen_LoRA': qwen_predict_proba,
}

# {dataset: {model: np.array of P(AI)}}
all_predictions = {}

for ds_name, ds_data in dataset_cache.items():
    print(f'\n{"="*60}')
    print(f'Dataset: {ds_name} ({len(ds_data["texts"])} samples)')
    print(f'{"="*60}')

    all_predictions[ds_name] = {}

    for model_name, predict_fn in MODEL_PREDICTORS.items():
        print(f'\n  Running {model_name}...')
        probs = predict_fn(ds_data['texts'])
        all_predictions[ds_name][model_name] = probs

        y_pred = (probs >= 0.5).astype(int)
        acc = (y_pred == ds_data['labels']).mean()
        print(f'    Mean P(AI): {probs.mean():.4f}, Acc: {acc:.4f}')

print('\nAll per-model predictions complete.')


Dataset: PAN2025 (3589 samples)

  Running LR...
    Mean P(AI): 0.6442, Acc: 0.9707

  Running DeBERTa...


    Mean P(AI): 0.5045, Acc: 0.8654

  Running Qwen_LoRA...


    Mean P(AI): 0.6433, Acc: 0.9975

Dataset: HC3 (1684 samples)

  Running LR...
    Mean P(AI): 0.5904, Acc: 0.5214

  Running DeBERTa...


    Mean P(AI): 0.0060, Acc: 0.5006

  Running Qwen_LoRA...


    Mean P(AI): 0.4873, Acc: 0.9869

Dataset: RAID (2000 samples)

  Running LR...
    Mean P(AI): 0.6568, Acc: 0.5005

  Running DeBERTa...


    Mean P(AI): 0.0638, Acc: 0.5490

  Running Qwen_LoRA...


    Mean P(AI): 0.3473, Acc: 0.8460

All per-model predictions complete.


In [25]:
# =====================================================
# CELL 9 — Save raw predictions
# =====================================================
predictions_to_save = {}
for ds_name, model_preds in all_predictions.items():
    predictions_to_save[ds_name] = {
        model: probs.tolist() for model, probs in model_preds.items()
    }

with open('/content/per_model_predictions.json', 'w') as f:
    json.dump(predictions_to_save, f)
print('Saved per-model predictions.')

Saved per-model predictions.


## Ensemble Strategies

In [26]:
# =====================================================
# CELL 10 — Define ensemble strategies
# =====================================================

# Weights based on avg OOD ROC-AUC from experiment 006
OOD_ROCAUC = {'LR': 0.5622, 'DeBERTa': 0.5909, 'Qwen_LoRA': 0.9067}
total = sum(OOD_ROCAUC.values())
W = {k: v / total for k, v in OOD_ROCAUC.items()}

print('Ensemble weights (normalized OOD ROC-AUC):')
for model, w in W.items():
    print(f'  {model}: {w:.4f}')


def ensemble_mean(preds):
    """Equal-weight probability averaging."""
    return np.stack(list(preds.values()), axis=0).mean(axis=0)


def ensemble_weighted(preds):
    """Weighted probability averaging."""
    result = np.zeros_like(list(preds.values())[0])
    for model_name, probs in preds.items():
        result += W[model_name] * probs
    return result


def ensemble_majority_vote(preds):
    """Hard majority vote at 0.5 threshold."""
    votes = np.stack([(p >= 0.5).astype(int) for p in preds.values()], axis=0)
    majority = (votes.sum(axis=0) >= 2).astype(int)
    # For ROC-AUC: use mean prob as score
    avg_prob = np.stack(list(preds.values()), axis=0).mean(axis=0)
    return majority, avg_prob

Ensemble weights (normalized OOD ROC-AUC):
  LR: 0.2729
  DeBERTa: 0.2869
  Qwen_LoRA: 0.4402


## Evaluate All Strategies

In [27]:
# =====================================================
# CELL 11 — Compute metrics for individuals + ensembles
# =====================================================
all_results = {}

for ds_name, ds_data in dataset_cache.items():
    all_results[ds_name] = {}
    labels = ds_data['labels']
    preds = all_predictions[ds_name]

    # --- Individual models ---
    for model_name, probs in preds.items():
        y_pred = (probs >= 0.5).astype(int)
        all_results[ds_name][model_name] = get_all_metrics(labels, y_pred, probs)

    # --- Ensemble: Mean ---
    p_mean = ensemble_mean(preds)
    y_pred = (p_mean >= 0.5).astype(int)
    all_results[ds_name]['Ensemble (Mean)'] = get_all_metrics(labels, y_pred, p_mean)

    # --- Ensemble: Weighted ---
    p_weighted = ensemble_weighted(preds)
    y_pred = (p_weighted >= 0.5).astype(int)
    all_results[ds_name]['Ensemble (Weighted)'] = get_all_metrics(labels, y_pred, p_weighted)

    # --- Ensemble: Majority Vote ---
    mv_labels, mv_prob = ensemble_majority_vote(preds)
    all_results[ds_name]['Ensemble (Majority Vote)'] = get_all_metrics(labels, mv_labels, mv_prob)

print('All evaluations complete.')

All evaluations complete.


## Results

In [28]:
# =====================================================
# CELL 12 — ROC-AUC comparison table
# =====================================================
rows = []
for ds_name, model_results in all_results.items():
    for model_name, metrics in model_results.items():
        row = {'Dataset': ds_name, 'Model': model_name}
        row.update(metrics)
        rows.append(row)

df_results = pd.DataFrame(rows)

# ROC-AUC pivot
roc_pivot = df_results.pivot(index='Model', columns='Dataset', values='ROC-AUC')
ood_cols = [c for c in roc_pivot.columns if c != 'PAN2025']
roc_pivot['Avg OOD'] = roc_pivot[ood_cols].mean(axis=1)

col_order = ['PAN2025', 'HC3', 'RAID', 'Avg OOD']
roc_pivot = roc_pivot[[c for c in col_order if c in roc_pivot.columns]]
roc_pivot = roc_pivot.sort_values('Avg OOD', ascending=False)

print('ROC-AUC Comparison (all models + ensembles)')
print('=' * 70)
print(roc_pivot.round(4).to_string())

ROC-AUC Comparison (all models + ensembles)
Dataset                   PAN2025     HC3    RAID  Avg OOD
Model                                                     
Qwen_LoRA                  0.9999  0.9982  0.8152   0.9067
Ensemble (Weighted)        0.9994  0.9919  0.8059   0.8989
Ensemble (Mean)            0.9994  0.9907  0.8028   0.8967
Ensemble (Majority Vote)   0.9994  0.9907  0.8028   0.8967
DeBERTa                    0.9948  0.5939  0.7351   0.6645
LR                         0.9953  0.5519  0.6074   0.5796


In [29]:
# =====================================================
# CELL 13 — Full metrics for each ensemble strategy
# =====================================================
metric_cols = ['ROC-AUC', 'Brier', 'F1', 'C@1', 'F0.5u']

for ds_name in dataset_cache.keys():
    print(f'\n{"="*60}')
    print(f'Dataset: {ds_name}')
    print(f'{"="*60}')

    ds_rows = df_results[df_results['Dataset'] == ds_name]
    print(ds_rows[['Model'] + metric_cols].set_index('Model').round(4).to_string())


Dataset: PAN2025
                          ROC-AUC   Brier      F1     C@1   F0.5u
Model                                                            
LR                         0.9953  0.0281  0.9774  0.9707  0.9736
DeBERTa                    0.9948  0.1106  0.8834  0.8654  0.9498
Qwen_LoRA                  0.9999  0.0019  0.9981  0.9975  0.9984
Ensemble (Mean)            0.9994  0.0211  0.9959  0.9947  0.9981
Ensemble (Weighted)        0.9994  0.0157  0.9972  0.9964  0.9986
Ensemble (Majority Vote)   0.9994  0.0211  0.9948  0.9933  0.9974

Dataset: HC3
                          ROC-AUC   Brier      F1     C@1   F0.5u
Model                                                            
LR                         0.5519  0.2705  0.6068  0.5214  0.5481
DeBERTa                    0.5939  0.4939  0.0024  0.5006  0.0059
Qwen_LoRA                  0.9982  0.0112  0.9868  0.9869  0.9947
Ensemble (Mean)            0.9907  0.1346  0.8354  0.8587  0.9269
Ensemble (Weighted)        0.9919  0.0989  0

In [30]:
# =====================================================
# CELL 14 — Delta: Best ensemble vs Qwen LoRA solo
# =====================================================
print('Delta: Best Ensemble vs Qwen LoRA (solo)')
print('=' * 60)

for ds_name in dataset_cache.keys():
    qwen_roc = all_results[ds_name]['Qwen_LoRA']['ROC-AUC']

    best_ens_name, best_ens_roc = None, -1
    for key, metrics in all_results[ds_name].items():
        if key.startswith('Ensemble'):
            if metrics['ROC-AUC'] > best_ens_roc:
                best_ens_roc = metrics['ROC-AUC']
                best_ens_name = key

    delta = best_ens_roc - qwen_roc
    marker = '+' if delta > 0 else '-'
    print(f'  {ds_name}: Qwen={qwen_roc:.4f}, {best_ens_name}={best_ens_roc:.4f}, d={delta:+.4f} [{marker}]')

Delta: Best Ensemble vs Qwen LoRA (solo)
  PAN2025: Qwen=0.9999, Ensemble (Weighted)=0.9994, d=-0.0005 [-]
  HC3: Qwen=0.9982, Ensemble (Weighted)=0.9919, d=-0.0063 [-]
  RAID: Qwen=0.8152, Ensemble (Weighted)=0.8059, d=-0.0093 [-]


In [31]:
# =====================================================
# CELL 15 — Save results
# =====================================================
from datetime import datetime

results_payload = {
    'experiment': '007-ensemble-baseline',
    'evaluated_at': datetime.now().isoformat(),
    'device': device,
    'models': ['LR (001)', 'DeBERTa (003)', 'Qwen LoRA (006)'],
    'ensemble_weights': W,
    'datasets': list(dataset_cache.keys()),
    'results': all_results,
}

with open('/content/ensemble_results.json', 'w') as f:
    json.dump(results_payload, f, indent=2)
print('Saved ensemble_results.json')

# Also save the ROC-AUC table as CSV
roc_pivot.to_csv('/content/roc_auc_comparison.csv')
print('Saved roc_auc_comparison.csv')

df_results.to_csv('/content/full_results.csv', index=False)
print('Saved full_results.csv')

print(f'\nDone. Download from /content/ to save locally.')

Saved ensemble_results.json
Saved roc_auc_comparison.csv
Saved full_results.csv

Done. Download from /content/ to save locally.


In [ ]:
from huggingface_hub import HfApi, login

login(token="hf_XXX")

api = HfApi()
api.upload_file(
    path_or_fileobj='/content/full_results.csv',
    path_in_repo='ensemble_evaluation_results.json',
    repo_id='hersheys-baklava/qwen-lora-pan2026',
    commit_message='Upload full ensemble evaluation results (PAN2025 + 5 OOD datasets)',
)

CommitInfo(commit_url='https://huggingface.co/hersheys-baklava/qwen-lora-pan2026/commit/fa0a9ef0d92508cf7fbb20ed12ad60a4ed9417c9', commit_message='Upload full ensemble evaluation results (PAN2025 + 5 OOD datasets)', commit_description='', oid='fa0a9ef0d92508cf7fbb20ed12ad60a4ed9417c9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hersheys-baklava/qwen-lora-pan2026', endpoint='https://huggingface.co', repo_type='model', repo_id='hersheys-baklava/qwen-lora-pan2026'), pr_revision=None, pr_num=None)